# Ch3 Self-Attention 教案

**课程名称：** Self-Attention：让词向量在上下文中"流动"

**预计总时长：** 90-100 分钟

**源文件：** `Ch3_Self_Attention/Ch3_Self_Attention.ipynb`（共 26 个 Cell，Cell 0-25）

---

## 时间表

| 时间段 | 内容 | Cell 范围 | 时长 |
|:---|:---|:---|:---|
| 00:00-05:00 | 开场与环境准备 | Cell 0-4 | 5 分钟 |
| 05:00-15:00 | 为什么需要 Attention + 历史演变 | Cell 5-6 | 10 分钟 |
| 15:00-30:00 | Q/K/V 直觉与手写 Attention | Cell 7-11 | 15 分钟 |
| 30:00-35:00 | 休息 + 回顾 | -- | 5 分钟 |
| 35:00-50:00 | 标准 Self-Attention（带投影矩阵） | Cell 12-13 | 15 分钟 |
| 50:00-65:00 | Masked Attention：因果掩码 | Cell 14-16 | 15 分钟 |
| 65:00-70:00 | 休息 + 回顾 | -- | 5 分钟 |
| 70:00-85:00 | Multi-Head Attention | Cell 17-19 | 15 分钟 |
| 85:00-92:00 | 真实文本演示 + 总结 | Cell 20-24 | 7 分钟 |
| 92:00-100:00 | 练习：手算 q2 注意力 | Cell 25 | 8 分钟 |

---

## 课前准备

- [ ] 确认 PyTorch 已安装（源 notebook 使用 PyTorch 2.10.0+cu128）
- [ ] 确认 matplotlib、seaborn、numpy 可用
- [ ] 确认中文字体设置正确（Microsoft YaHei / SimHei）
- [ ] 提前运行一遍全部 Cell，确认无报错
- [ ] 准备好 `attention_alammar.png` 图片文件在 Ch3 目录下
- [ ] 打开 Jay Alammar 的 Illustrated Transformer 页面备用
- [ ] 准备白板/画板用于画 Q/K/V 流程图

---

## 第一段：开场与环境准备（Cell 0-4）

📍 运行 Cell 0-4（Cell 0-3 为 Markdown，Cell 4 为代码）

⏱ 时间分配：5 分钟

🎯 本段目标
- 建立学习动机：静态 Embedding 的局限
- 确认环境就绪
- 让学生对本章结构有全局感

🗣 讲课话术

> 大家好，上一章我们学了 Embedding，把每个词变成一个向量。但我想问大家一个问题——"苹果很好吃"和"苹果发布新手机"，这两个句子里的"苹果"应该是同一个向量吗？
>
> 显然不应该对吧？一个是水果，一个是公司。但静态 Embedding 只有一个固定向量，区分不了。这就是我们今天要解决的问题。
>
> Self-Attention 的核心思想特别简单：让每个词"看看"周围的词，然后根据上下文调整自己的表示。苹果旁边是"好吃"，它就偏向水果；旁边是"发布""手机"，它就偏向公司。
>
> 我们先把环境跑起来。运行 Cell 4，确认 PyTorch 版本输出正常。

👀 输出要点
- Cell 4 应输出：`PyTorch version: 2.10.0+cu128`
- 如果版本不同不影响运行，只要 >= 1.9 即可

❓ 预判问题

Q: 为什么不用 TensorFlow？
A: PyTorch 在研究和教学中更主流，代码更直观。核心概念是通用的，框架只是工具。

Q: Cell 2 提到的"动态向量"是什么意思？
A: 就是同一个词在不同上下文中得到不同的向量表示。这正是 Self-Attention 的作用，我们马上就会看到。

➡️ 转场

> 好，环境没问题。接下来我们先从理论上理解：Attention 到底是怎么来的？它的核心公式长什么样？

---

## 第二段：为什么需要 Attention + 历史演变（Cell 5-6）

📍 运行 Cell 5-6（均为 Markdown，无需运行代码）

⏱ 时间分配：10 分钟

🎯 本段目标
- 理解 Attention 的历史脉络（Bahdanau → Luong → Vaswani）
- 掌握核心公式 Attention(Q,K,V) = softmax(QK^T / sqrt(d_k)) V
- 理解为什么要除以 sqrt(d_k)（缩放因子的严格推导）

🗣 讲课话术

> 在讲代码之前，我们先花几分钟了解 Attention 的来龙去脉。
>
> 2014 年，Bahdanau 做机器翻译时遇到一个问题：Seq2Seq 模型把整个源句压缩成一个固定向量，长句信息丢失严重。他的解决方案是让解码器在生成每个词时"回头看"编码器的所有隐状态——这就是最早的 Attention。
>
> 2015 年 Luong 用点积替代了加法，计算更快。然后 2017 年 Vaswani 的"Attention Is All You Need"直接去掉了 RNN，完全用 Attention 来构建模型——这就是 Transformer 的诞生。
>
> 现在看核心公式：Attention(Q,K,V) = softmax(QK^T / sqrt(d_k)) * V。
>
> 这里有个关键问题：**为什么要除以 sqrt(d_k)？** 这是面试高频题。
>
> 我给大家一个直觉：假设 d_k = 64，Q 和 K 的元素都是标准正态分布。点积是 64 个随机数乘积之和，方差就是 64，标准差就是 8。这意味着点积值可能跑到 [-16, +16] 的范围。
>
> softmax 对这么大的输入会怎样？它会变成几乎是 one-hot 的分布——比如 softmax([-16, 0, 16]) 基本上就是 [0, 0, 1]。梯度几乎为零，模型学不动了！
>
> 除以 sqrt(64) = 8 之后呢？[-16, 0, 16] 变成 [-2, 0, 2]，softmax 输出大约是 [0.09, 0.24, 0.67]，分布平滑，梯度充足。
>
> 还有个有趣的联系：这个缩放因子本质上是一个"温度参数"。低温度 = 硬注意力（只看最相关的），高温度 = 均匀注意力（什么都看）。GPT 生成文本时调节的 temperature 就是这个东西。

👀 输出要点
- 本段为纯理论 Markdown，无代码输出
- 重点讲解 Cell 6 中的缩放因子推导表格和温度参数表格
- Cell 6 中的复杂度分析表：QK^T 时间 O(n^2 d)、空间 O(n^2)

❓ 预判问题

Q: Self-Attention 和 Cross-Attention 有什么区别？
A: Self-Attention 的 Q/K/V 来自同一个序列（如 GPT 每一层）；Cross-Attention 的 Q 来自解码器、K/V 来自编码器（如机器翻译）。Cell 6 有对照表。

Q: 计算复杂度 O(n^2 d) 有多严重？
A: 序列长度翻倍，计算量翻 4 倍。GPT-2 上下文 1024 tokens 需要约 100 万个注意力分数；GPT-4 的 128K tokens 就是 164 亿个。这就是为什么 FlashAttention 等优化很重要。

Q: 温度参数和 ChatGPT 的 temperature 设置是一回事吗？
A: 原理一样！ChatGPT 的 temperature 控制的是输出 token 采样时的 softmax 锐度。低温度更确定性，高温度更随机。

➡️ 转场

> 理论讲完了，来看代码。我们先用一个图书馆的比喻来直观理解 Q/K/V，然后亲手写 Attention。

---

## 第三段：Q/K/V 直觉与手写 Attention（Cell 7-11）

📍 运行 Cell 7（Q/K/V 图书馆比喻可视化）、Cell 8-9（Alammar 图 + 计算过程 Markdown）、Cell 10（手写 Attention 代码）、Cell 11（注意力权重热力图）

⏱ 时间分配：15 分钟（可视化 3 分钟 + Alammar 图讲解 3 分钟 + 代码 5 分钟 + 热力图 4 分钟）

🎯 本段目标
- 通过图书馆比喻理解 Q/K/V 的直觉含义
- 跟随 Alammar 的可视化理解 scores → z 的完整流程
- 亲手实现（或补全）4 步 Attention 代码
- 通过热力图验证注意力权重是概率分布

🗣 讲课话术

> 先运行 Cell 7 看一个可视化。大家看左边的图——把 Attention 想象成去图书馆找书。
>
> **Query** 就是你心里想的："我要找机器学习的书"。**Key** 是书架上每本书的标签："深度学习""Python编程""统计学"。**Value** 是书的实际内容。
>
> 你拿 Query 去和每个 Key 比对——"深度学习"和"机器学习"很相关，匹配度 0.95；"Python编程"关系不大，匹配度 0.3。然后按匹配度加权，把相关书的内容混合在一起，得到你要的答案。
>
> 中间的图展示了数学过程：Step 1 Q 乘 K 转置得到分数，Step 2 softmax 归一化为概率，Step 3 用概率加权 V。
>
> 现在看 Cell 8 的 Alammar 图和 Cell 9 的详细计算。这里用了一个具体数值例子：d_k=64，x1 每维都是 1.323，x2 每维都是 1.134。q1 和 k1 的点积 = 64 * 1.323 * 1.323 约等于 112，除以 sqrt(64)=8 后是 14。最终 softmax([14, 12]) 得到 [0.881, 0.119]。
>
> 好，现在到大家动手的时候了！Cell 10 是手写 Attention 的代码，有 4 个 TODO 要补全。
>
> 代码的核心就四步：
> 1. `scores = Q @ K^T`——注意 K 要转置最后两维，用 `K.transpose(-2, -1)`
> 2. `scores = scores / sqrt(d_k)`——缩放
> 3. `attention_weights = F.softmax(scores, dim=-1)`——对最后一维做 softmax
> 4. `output = attention_weights @ V`——加权求和

👀 输出要点
- Cell 7：三张并排图（图书馆比喻、数学过程、核心洞察），底部输出 Key takeaways
- Cell 10 输出：输入形状 `torch.Size([1, 5, 8])`，输出形状 `torch.Size([1, 5, 8])`，注意力权重形状 `torch.Size([1, 5, 5])`
- Cell 11 输出：5x5 热力图，每行和为 1。底部打印 `tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000])`
- 重点指出：输入 [1, 5, 8] 表示 1 个样本、5 个 token、8 维；权重 [1, 5, 5] 表示每个 token 对其他 5 个 token 的关注度

❓ 预判问题

Q: 为什么这里 Q=K=V=x，不需要投影矩阵吗？
A: 这是最简版本，用来理解核心计算流程。下一段 Cell 13 会加上 W_q/W_k/W_v 投影矩阵，那才是标准实现。

Q: `K.transpose(-2, -1)` 是什么意思？
A: 交换最后两个维度。K 原来是 [batch, seq_len, d_model]，转置后变成 [batch, d_model, seq_len]，这样 Q @ K^T 才能算出 [batch, seq_len, seq_len] 的分数矩阵。

Q: 热力图里每行和为 1，为什么？
A: 因为我们对最后一维（key 维度）做了 softmax，每行就是一个概率分布，表示当前 query 对所有 key 的关注度分配。

➡️ 转场

> 很好，最简版 Attention 我们写出来了。但真实模型里，Q/K/V 不是直接用 x，而是用投影矩阵把 x 映射到不同的空间。下面我们来看标准实现。

---

## 休息 + 回顾（第 30-35 分钟）

⏱ 时间分配：5 分钟

**三句话回顾前半段：**

1. 静态 Embedding 不能区分一词多义，Self-Attention 通过让词"看看"上下文来动态调整表示。
2. 核心公式四步走：Q 乘 K 转置 → 除以 sqrt(d_k) 缩放 → softmax 归一化 → 乘以 V 加权求和。
3. 除以 sqrt(d_k) 是为了防止点积过大导致 softmax 饱和（梯度消失），d_k=64 时标准差从 8 降到 1。

**下一段预告：**

> 接下来我们给 Attention 加上投影矩阵和 Mask，看看 GPT 为什么不能"偷看未来"。

---

## 第四段：标准 Self-Attention 实现（Cell 12-13）

📍 运行 Cell 13（SelfAttention 类定义 + 测试 + 因果掩码可视化）

⏱ 时间分配：15 分钟

🎯 本段目标
- 理解 W_q/W_k/W_v/W_o 四个投影矩阵的作用
- 理解 `masked_fill` 如何实现掩码
- 计算参数量并验证

🗣 讲课话术

> 上面我们直接让 Q=K=V=x，但这样有个问题——Q、K、V 的"角色"完全一样，模型没有灵活性。
>
> 标准做法是加四个 `nn.Linear` 投影矩阵：W_q 把 x 映射到"查询空间"，W_k 映射到"键空间"，W_v 映射到"值空间"，最后 W_o 把输出投影回原始维度。
>
> 大家看 Cell 13 的 `SelfAttention` 类。`__init__` 里定义了四个 Linear 层，`forward` 的流程和我们手写的完全一样，只是前面多了投影步骤。
>
> 注意 `mask` 参数：`scores.masked_fill(mask == 0, float('-inf'))`。把不该看的位置填上负无穷，softmax 之后这些位置的权重就自动变成 0。这个技巧非常优雅。
>
> 看输出：输入 `[2, 10, 32]`（2 个句子、10 个词、32 维），输出形状不变。参数量是 2048——大家可以验算一下：`W_q` 是 32x16=512，`W_k/W_v` 也各 512，`W_o` 是 16x32=512，加上 bias... 等等，这里用了 `bias=False` 所以就是 512*4=2048。
>
> Cell 13 下方还生成了一个因果掩码的可视化：6x6 的绿色热力图，下三角是 1（可看），上三角是 0（不可看）。

👀 输出要点
- Cell 13 文本输出：`输入: torch.Size([2, 10, 32])`、`输出: torch.Size([2, 10, 32])`、`参数量: 2048`
- Cell 13 图像输出：6x6 因果掩码热力图（绿色，下三角 1，上三角 0）
- 参数量计算：W_q(32x16) + W_k(32x16) + W_v(32x16) + W_o(16x32) = 512*4 = 2048，无 bias

❓ 预判问题

Q: 为什么 W_q 的 bias=False？
A: 原始 Transformer 论文中 Attention 投影矩阵不加 bias。实际上加不加 bias 影响很小，但不加可以减少参数和计算量。

Q: d_k 和 d_model 可以不同吗？
A: 可以！d_k 是注意力的"工作维度"，通常设为 d_model / n_heads。这里 d_model=32、d_k=16，就是投影到更低维度来计算注意力。

Q: `masked_fill` 为什么用负无穷而不是 0？
A: 因为是在 softmax 之前填充。exp(-inf) = 0，所以 softmax 后这些位置的权重自然变成 0。如果直接在 scores 上填 0，softmax 后还是正数，起不到掩码效果。

➡️ 转场

> 我们刚才看到了因果掩码的形状，接下来深入讲讲为什么语言模型需要这个 Mask——为什么不能"偷看答案"。

---

## 第五段：Masked Attention——因果掩码（Cell 14-16）

📍 运行 Cell 15（create_causal_mask 函数 + 可视化）、Cell 16（有/无 Mask 对比）

⏱ 时间分配：15 分钟

🎯 本段目标
- 理解为什么自回归语言模型需要因果掩码
- 掌握 `torch.tril` 创建下三角矩阵
- 通过对比图直观看到掩码的效果

🗣 讲课话术

> 大家想想 GPT 生成文本的过程：它一个词一个词地生成，生成第 3 个词的时候，第 4、5 个词还不存在。如果训练的时候让它看到了未来的词，那不就是"开卷考试"吗？模型学到的就不是真正的语言规律，而是"抄答案"。
>
> 所以我们需要因果掩码（Causal Mask）。看 Cell 15，`torch.tril` 生成下三角矩阵：t=0 时只能看自己，t=1 能看 t=0 和 t=1，t=5 能看所有位置。
>
> 现在看 Cell 16 的对比图，这是最直观的！左边是无掩码的双向 Attention——每个位置都能看到所有位置，权重分布在整个矩阵。右边是有因果掩码的单向 Attention——上三角全是 0，每个位置只能关注它之前和自己的位置。
>
> 注意一个细节：有 Mask 后，下三角部分的权重值变大了。因为原来分配给上三角的"注意力预算"现在全部重新分配到了下三角。每行的权重和仍然是 1。
>
> 这里有个重要的区分：BERT 用的是双向 Attention（无掩码），所以 BERT 擅长理解任务；GPT 用的是单向 Attention（有因果掩码），所以 GPT 擅长生成任务。

👀 输出要点
- Cell 15：6x6 绿色因果掩码热力图 + 文字解释（t=0 只看自己，t=5 看所有位置）
- Cell 16：并排两张蓝色热力图。左图（无掩码）权重分布在全矩阵；右图（有掩码）上三角为 0.00
- Cell 16 底部输出：`观察：有 Mask 时，上三角部分权重为 0（不能偷看未来）`

❓ 预判问题

Q: BERT 不需要 Mask 吗？
A: BERT 是双向模型，训练目标是 Masked Language Model（完形填空），所以确实不需要因果掩码。但 BERT 有另一种 Mask——padding mask，用来忽略填充的 token。

Q: 因果掩码会不会导致模型"视野太窄"？
A: 单层确实如此，但 Transformer 是多层堆叠的。深层的 Attention 可以间接看到更远的信息——第 2 层的 token 看的是第 1 层已经融合了上下文的表示。

Q: 生成时不需要 Mask 了吧？
A: 生成时本来就只有已生成的 token，物理上就看不到未来。但训练时所有 token 是同时输入的，所以必须用 Mask 模拟逐步生成的过程。

➡️ 转场

> Attention 的基本版本我们都掌握了。但一个注意力头只能学一种"关注模式"。如果我们想同时关注句法、语义、位置等多种关系怎么办？答案是——多头注意力。

---

## 休息 + 回顾（第 65-70 分钟）

⏱ 时间分配：5 分钟

**三句话回顾：**

1. 标准 Self-Attention 用 W_q/W_k/W_v/W_o 四个投影矩阵把输入映射到不同空间，参数量为 2048（d_model=32, d_k=16 时）。
2. 因果掩码用下三角矩阵实现，在 softmax 之前把上三角填为负无穷，确保每个位置只看过去。
3. 双向 Attention（BERT）能看所有位置，单向 Attention（GPT）只看过去——这决定了模型适合理解还是生成。

**下一段预告：**

> 接下来是本章最精彩的部分——多头注意力。8 个头并行工作，每个头学习不同的"关注模式"。

---

## 第六段：Multi-Head Attention（Cell 17-19）

📍 运行 Cell 18（MultiHeadAttention 类 + 测试）、Cell 19（8 个头的注意力模式可视化）

⏱ 时间分配：15 分钟

🎯 本段目标
- 理解多头注意力的动机：不同的头学习不同的关注模式
- 掌握"分头 = reshape + transpose"的实现技巧
- 理解参数量不变的数学关系：h x d_k = d_model
- 通过 8 头可视化直观看到不同头的模式差异

🗣 讲课话术

> 我先问大家一个问题："The cat sat on the mat because it was tired"。要理解这句话，你需要同时捕捉哪些关系？
>
> "cat"是"sat"的主语——这是句法关系；"it"指代"cat"而不是"mat"——这是指代关系；"tired"和"cat"语义相关——这是语义关系。
>
> 一个注意力头只能产生一种注意力分布（一个概率向量），强迫一个头同时捕捉这么多关系，它只能做折中。所以我们用多个头，每个头专注一种模式。
>
> Cell 17 的理论部分有一个关键洞察：**多头不增加总计算量**。d_model=512、8 个头时，每个头的 d_k = 512/8 = 64。单头的 W_q 是 512x512，参数量 3x512^2。多头 W_q 仍然是 512x512（包含所有头），参数量一样！
>
> 现在看代码 Cell 18。关键在于"分头"操作——不是真的创建 8 个独立的矩阵，而是一个大矩阵投影后 reshape。看这行：
> `Q.view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)`
> 从 [batch, seq_len, d_model] 变成 [batch, n_heads, seq_len, d_k]，所有头的注意力可以在一次矩阵乘法中并行完成。
>
> 看输出：输入 [2, 10, 64]，输出 [2, 10, 64]，注意力权重 [2, 8, 10, 10]——8 个头，每个头一个 10x10 的注意力矩阵。
>
> 现在运行 Cell 19 看最直观的效果——8 个头的注意力热力图。每个头的模式确实不同！有的比较集中，有的比较分散。在真实训练后的模型里，这些差异会更明显。

👀 输出要点
- Cell 18 输出：`输入: torch.Size([2, 10, 64])`、`输出: torch.Size([2, 10, 64])`、`注意力权重: torch.Size([2, 8, 10, 10])  (8个头，每个头10x10)`
- Cell 19 输出：2x4 排列的 8 张蓝色热力图，标题"注意力头 1"到"注意力头 8"，底部打印 `观察：不同的头学习了不同的注意力模式！`
- 强调：这里是随机初始化，训练后差异会更显著

❓ 预判问题

Q: 头的数量怎么选？
A: 通常是 8 或 16。原始 Transformer（d_model=512）用 8 头，GPT-3（d_model=12288）用 96 头。关键约束是 d_model 能被 n_heads 整除。

Q: 最后的 W_o 有什么作用？
A: 多个头的输出 concat 后维度是 d_model，W_o 负责"融合"不同头的信息。没有 W_o 的话，各个头的输出只是简单拼接，缺少交互。

Q: `.contiguous()` 是什么意思？
A: transpose 操作不会物理移动内存中的数据（只改变了步长信息），但 view 需要连续内存。`.contiguous()` 确保数据在内存中是连续存放的。

➡️ 转场

> 到目前为止我们都用随机数据。最后让我们在一个真实的英语句子上跑一下 Attention，看看效果。

---

## 第七段：真实文本演示 + 总结（Cell 20-24）

📍 运行 Cell 21（"The cat sat on the mat" Attention 演示）、浏览 Cell 22-24（总结 Markdown）

⏱ 时间分配：7 分钟

🎯 本段目标
- 在真实文本上看到 Attention 权重的分布
- 总结全章核心概念
- 引出下一章 Transformer Block

🗣 讲课话术

> 最后我们用 "The cat sat on the mat" 这句话来跑一下 Attention。运行 Cell 21。
>
> 注意这里用的是**随机嵌入**，所以注意力模式不反映真实语义关系——这只是演示计算流程。真实模型训练之后，你会看到 "cat" 和 "sat" 之间的权重更高（主谓关系），两个 "the" 之间也可能有较高权重（句法模式）。
>
> 热力图的颜色越深表示关注度越高。每一行代表一个 query 词关注所有 key 词的程度。
>
> 现在看 Cell 22 的总结。那个 ASCII 流程图把整个 Self-Attention 的数据流画得非常清楚：输入 X → 三个投影 → Q/K/V → 点积 → 缩放 → 可选 Mask → softmax → 乘以 V → 输出。Multi-Head 就是把这个流程并行跑 h 次再拼接。
>
> Cell 22 还有四道面试题，大家课后一定要看，尤其是"为什么除以 sqrt(d_k)"——这个几乎百分百会考。
>
> 下一章我们会把 Attention 作为一个组件，加上残差连接、LayerNorm 和 FFN，组装出完整的 Transformer Block。

👀 输出要点
- Cell 21：6x6 黄红色热力图，行标签/列标签都是 ["The", "cat", "sat", "on", "the", "mat"]
- Cell 21 底部：`注意：此处使用随机嵌入，注意力模式不反映真实语义关系，仅演示计算流程`
- Cell 22：核心概念图谱、关键公式速查表、四道面试题

❓ 预判问题

Q: 为什么不用真实的预训练 Embedding？
A: 为了让代码简洁聚焦于 Attention 机制本身。真实应用中会用 `nn.Embedding` 或预训练权重，在后续章节中会整合。

Q: 面试时怎么快速回答"为什么除以 sqrt(d_k)"？
A: 一句话版本："点积的方差随 d_k 线性增长，除以 sqrt(d_k) 将方差归一化为 1，防止 softmax 饱和导致梯度消失。"

➡️ 转场

> 最后留几分钟给大家做一个手算练习——Cell 25 的 q2 注意力计算。这个练习能帮你真正内化整个 Attention 流程。

---

## 第八段：练习——手算 q2 注意力（Cell 23, 25）

📍 运行 Cell 25（手算 q2 注意力验证代码）

⏱ 时间分配：8 分钟

🎯 本段目标
- 学生独立完成手算 q2 的 4 步 Attention 计算
- 用代码验证手算结果

🗣 讲课话术

> Cell 23 的 Extra 部分有一个手算练习。设 d_k=64，x1 每维 1.323，x2 每维 1.134，Q=K=V=x。
>
> 大家先自己算 q2 对 k1 和 k2 的注意力分数，算完了再运行 Cell 25 验证。
>
> 我给两分钟时间，先自己试。

### Hint 节奏

**0-2 分钟：** 自己尝试，不给提示。

**2 分钟第一个提示：**
> q2 和 k1 的点积 = d_k * x2_val * x1_val = 64 * 1.134 * 1.323。记住 Q=K=V=x，所以 q2 就是 x2。

**4 分钟关键代码：**
> - q2·k1 = 64 * 1.134 * 1.323 = 96.02
> - q2·k2 = 64 * 1.134 * 1.134 = 82.30
> - scaled: [96.02/8, 82.30/8] = [12.00, 10.29]
> - softmax([12.00, 10.29]) = [0.847, 0.153]
> - z2 = 0.847 * 1.323 + 0.153 * 1.134 = 1.2942

### 常见错误

1. **忘记缩放**：直接对 [96.02, 82.30] 做 softmax，得到几乎是 [1, 0] 的分布（正好验证了为什么要缩放）
2. **sqrt 搞错**：用 d_k=64 而不是 sqrt(64)=8 去除
3. **q2·k2 算错**：用 x1_val * x2_val 而不是 x2_val * x2_val（q2 对应 x2，k2 也对应 x2）

### 验证标准

运行 Cell 25 后应看到：
- `Step 1: q2·k1 = 96.02, q2·k2 = 82.30`
- `Step 2: scaled scores = [12.00, 10.29]`
- `Step 3: attention weights = [0.847, 0.153]`
- `Step 4: z2 = 0.847 * 1.323 + 0.153 * 1.134 = 1.2942`
- `z2 = [1.2942, 1.2942, ..., 1.2942]  (64 维)`

👀 输出要点
- Cell 25 完整输出如上所列
- 对比 q1 的结果（Cell 9 中提到的 z1 = [1.300, ...]），q2 的结果 z2 = [1.2942, ...] 更接近 x2 自己的值 1.134，因为 q2 对 k2 的权重（0.153）比 q1 对 k2 的权重（0.119）更大

❓ 预判问题

Q: 为什么 q2 更关注 k1 而不是 k2？
A: 因为 x1 的值（1.323）比 x2（1.134）大，在这个简化例子中点积主要取决于向量的"大小"。真实模型里有投影矩阵，方向（而非大小）才是关键。

Q: 64 维向量每维都一样，这现实吗？
A: 不现实，这只是为了方便手算。真实 Embedding 的每一维都不同，但核心计算流程完全一样。

---

## 附录 A：时间快速参考表

| 分钟 | 事件 | Cell |
|:---|:---|:---|
| 0 | 开场，运行环境准备 | 0-4 |
| 5 | 讲解 Attention 动机与历史 | 5-6 |
| 15 | Q/K/V 可视化 + 手写 Attention | 7-11 |
| 30 | **休息** | -- |
| 35 | 标准 Self-Attention 类 | 12-13 |
| 50 | Masked Attention + 对比 | 14-16 |
| 65 | **休息** | -- |
| 70 | Multi-Head Attention + 8头可视化 | 17-19 |
| 85 | 真实文本演示 + 总结 | 20-24 |
| 92 | 手算练习 q2 | 23, 25 |
| 100 | 结束 | -- |

---

## 附录 B：关键数据快速参考

### 核心公式

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

### 张量维度速查

| 变量 | 形状 | 说明 |
|:---|:---|:---|
| x (输入) | [batch, seq_len, d_model] | Cell 10: [1, 5, 8] |
| Q, K, V | [batch, seq_len, d_k] | 投影后可能维度不同 |
| scores | [batch, seq_len, seq_len] | Cell 10: [1, 5, 5] |
| attention_weights | [batch, seq_len, seq_len] | 每行和为 1 |
| output | [batch, seq_len, d_model] | 维度与输入相同 |
| MHA weights | [batch, n_heads, seq_len, seq_len] | Cell 18: [2, 8, 10, 10] |

### 关键数值

| 数值 | 来源 | 含义 |
|:---|:---|:---|
| 参数量 2048 | Cell 13 (d_model=32, d_k=16) | 4 个 Linear 无 bias |
| 缩放因子 sqrt(64) = 8 | Cell 6/9/25 | d_k=64 时的缩放 |
| softmax([-2,0,2]) = [0.09, 0.24, 0.67] | Cell 6 理论 | 缩放后的平滑分布 |
| softmax([-16,0,16]) ≈ [0, 0, 1] | Cell 6 理论 | 未缩放的饱和分布 |
| q1 权重 [0.881, 0.119] | Cell 9 | Alammar 例子 |
| q2 权重 [0.847, 0.153] | Cell 25 | 练习答案 |
| z1 = 1.300, z2 = 1.2942 | Cell 9/25 | 加权输出值 |

---

## 附录 C：应急预案

### 场景 1：环境问题

**症状：** Cell 4 报错 `ModuleNotFoundError: No module named 'torch'`

**应对：**
1. 在终端运行 `pip install torch torchvision`
2. 如果是 conda 环境：`conda install pytorch -c pytorch`
3. 备选：切换到 Google Colab，上传 notebook

### 场景 2：中文字体不显示

**症状：** matplotlib 图表中文显示为方框

**应对：**
1. Cell 4 已配置了 `plt.rcParams["font.sans-serif"]` 备选字体列表
2. 如果仍有问题，在 Cell 4 后插入：`plt.rcParams['font.sans-serif'] = ['DejaVu Sans']`（牺牲中文，保证图能看）
3. 口头补充中文标签含义

### 场景 3：Cell 10 学生卡住（TODO 补全）

**应对：**
1. 源 notebook 已包含答案（TODO 后面有参考实现）
2. 如果学生需要从零写，先给框架提示："用 torch.matmul 和 K.transpose(-2,-1)"
3. 最多 3 分钟后展示答案，不要卡在这里太久

### 场景 4：时间不够

**可跳过的内容（按优先级）：**
1. Cell 7 的 Q/K/V 可视化图（口头讲解即可）- 省 2 分钟
2. Cell 21 真实文本演示（随机 Embedding 意义有限）- 省 3 分钟
3. Cell 9 详细计算过程（改为快速口述）- 省 3 分钟

**不可跳过的核心：**
- Cell 10-11：手写 Attention + 热力图（核心动手环节）
- Cell 15-16：因果掩码对比（理解 GPT 的关键）
- Cell 18-19：多头 Attention + 可视化（本章最重要的扩展）

### 场景 5：学生提出超纲问题（如 FlashAttention、KV Cache）

**应对：**
1. 简要回答核心思想（如 FlashAttention = 分块计算避免存储完整 n x n 矩阵）
2. 记录在白板上，承诺后续章节或课后讨论
3. 不要展开，以免偏离主线